# Modül 04: Geri Yayılım (Backpropagation)
## Deep Learning Path — Kapsamlı Jupyter Notebook
---
## 📋 İçindekiler
1. [Öğrenme Hedefleri](#1)
2. [Zincir Kuralı — Teori ve Sayısal Doğrulama](#2)
3. [Hesaplama Grafı](#3)
4. [Tam MLP Backpropagation](#4)
5. [Gradyan Kontrolü](#5)
6. [Mini Autograd Motoru](#6)
7. [Vanishing & Exploding Gradient](#7)
8. [TF / PyTorch Autograd Karşılaştırması](#8)
9. [Hiperparametre Deneyleri](#9)
10. [Özet ve Alıştırmalar](#10)


## 1. Öğrenme Hedefleri <a id='1'></a>
- ✅ Zincir kuralını çok değişkenli fonksiyonlara uygulamak
- ✅ Hesaplama grafı çizerek forward/backward pass'i takip etmek
- ✅ 2 katmanlı MLP için tüm gradyanları adım adım hesaplamak
- ✅ Gradyan kontrolüyle backprop implementasyonunu doğrulamak
- ✅ Sıfırdan mini autograd motoru (Value sınıfı) yazmak
- ✅ Vanishing ve exploding gradient'ı görsel kanıtlarıyla açıklamak
- ✅ Gradient clipping implementasyonu yazmak


## 2. Zincir Kuralı <a id='2'></a>

Eğer $z = f(y)$ ve $y = g(x)$ ise:
$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}$$

Derin ağda $L = \text{BCE}(\sigma(W_2 \cdot \sigma(W_1 x + b_1) + b_2),\ y)$:

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial A_2} \cdot \frac{\partial A_2}{\partial Z_2} \cdot \frac{\partial Z_2}{\partial A_1} \cdot \frac{\partial A_1}{\partial Z_1} \cdot \frac{\partial Z_1}{\partial W_1}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

# Sayısal gradyan (merkezi fark) vs analitik
def f(x):       return x**3 + 2*x**2 - 5*x + 1
def f_grad(x):  return 3*x**2 + 4*x - 5  # analitik
def num_grad(fn, x, h=1e-5): return (fn(x+h)-fn(x-h))/(2*h)

print("f(x) = x³ + 2x² − 5x + 1   →   f'(x) = 3x² + 4x − 5")
print(f"{'x':>5} | {'Analitik':>12} | {'Sayısal':>12} | {'Fark':>12}")
print("-"*47)
for x in [-2.,-1.,0.,1.,2.,3.]:
    a = f_grad(x); n = num_grad(f,x)
    print(f"{x:>5.1f} | {a:>12.6f} | {n:>12.6f} | {abs(a-n):>12.2e}")
print("\n✓ Farklar 1e-10 düzeyinde — analitik türev doğrulandı")


## 3. Hesaplama Grafı <a id='3'></a>

```
L = (w·x − y)²  için:

FORWARD:   x=3, w=2, y=5
  x=3 ──┐
         ├──[×]── z=6 ──[−]── e=1 ──[²]── L=1
  w=2 ──┘              ↑
                        y=5

BACKWARD:
  dL/dL = 1
  dL/de = 2e = 2
  dL/dz = 2·1 = 2
  dL/dw = dL/dz·x = 2·3 = 6
  dL/dx = dL/dz·w = 2·2 = 4
```


In [ ]:
# Hesaplama grafı manuel doğrulama
w,x,y = 2.,3.,5.
z = w*x; e = z-y; L = e**2

dL_de = 2*e
dL_dz = dL_de*1.
dL_dw = dL_dz*x
dL_dx = dL_dz*w

print(f"L = ({w}·{x} − {y})² = {L}")
print(f"dL/dw = {dL_dw} | dL/dx = {dL_dx}")

# Sayısal doğrulama
nw = num_grad(lambda ww: (ww*x-y)**2, w)
nx = num_grad(lambda xx: (w*xx-y)**2, x)
print(f"Sayısal dL/dw = {nw:.4f} ✓")
print(f"Sayısal dL/dx = {nx:.4f} ✓")


## 4. Tam MLP Backpropagation <a id='4'></a>

### Forward Pass
$$Z_1 = XW_1+b_1 \quad A_1=\sigma(Z_1) \quad Z_2=A_1W_2+b_2 \quad \hat{y}=\sigma(Z_2)$$

### Backward Pass
$$\delta_2 = \frac{A_2-y}{n} \quad \nabla W_2 = A_1^T\delta_2 \quad \delta_1 = \delta_2 W_2^T \odot A_1(1-A_1) \quad \nabla W_1 = X^T\delta_1$$


In [ ]:
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))

class MLP:
    def __init__(self,ni,nh,no,lr=.5):
        self.lr=lr
        self.W1=np.random.randn(ni,nh)*np.sqrt(2/(ni+nh))
        self.b1=np.zeros((1,nh))
        self.W2=np.random.randn(nh,no)*np.sqrt(2/(nh+no))
        self.b2=np.zeros((1,no))
        self.cache={}

    def forward(self,X):
        Z1=X@self.W1+self.b1; A1=sigmoid(Z1)
        Z2=A1@self.W2+self.b2; A2=sigmoid(Z2)
        self.cache={'X':X,'A1':A1,'A2':A2}
        return A2

    def loss(self,y,yh):
        eps=1e-15; yh=np.clip(yh,eps,1-eps)
        return -np.mean(y*np.log(yh)+(1-y)*np.log(1-yh))

    def backward(self,y):
        n=y.shape[0]; X=self.cache['X']
        A1=self.cache['A1']; A2=self.cache['A2']
        dZ2=(A2-y)/n
        dW2=A1.T@dZ2; db2=np.sum(dZ2,axis=0,keepdims=True)
        dZ1=(dZ2@self.W2.T)*(A1*(1-A1))
        dW1=X.T@dZ1;  db1=np.sum(dZ1,axis=0,keepdims=True)
        self.W2-=self.lr*dW2; self.b2-=self.lr*db2
        self.W1-=self.lr*dW1; self.b1-=self.lr*db1
        return {'dW1':dW1,'dW2':dW2}

X_xor = np.array([[0,0],[0,1],[1,0],[1,1]],dtype=float)
y_xor = np.array([[0],[1],[1],[0]],dtype=float)

model = MLP(2,4,1,.5)
hist=[]
for _ in range(5000):
    yh=model.forward(X_xor); hist.append(model.loss(y_xor,yh)); model.backward(y_xor)

preds=(model.forward(X_xor)>=.5).astype(int)
print(f"Final Loss: {hist[-1]:.6f} | Accuracy: {np.mean(preds==y_xor):.1%}")


## 5. Gradyan Kontrolü <a id='5'></a>

**Altın kural:** Backprop implementasyonunu her zaman sayısal gradyanla doğrula. Fark < 1e-5 olmalı.

In [ ]:
def gradient_check(model, X, y, h=1e-5):
    yh = model.forward(X)
    n  = y.shape[0]
    A1,A2 = model.cache['A1'],model.cache['A2']
    dZ2=(A2-y)/n; dZ1=(dZ2@model.W2.T)*(A1*(1-A1))
    analytic = X.T @ dZ1

    num = np.zeros_like(model.W1)
    it  = np.nditer(model.W1,flags=['multi_index'])
    while not it.finished:
        idx=it.multi_index; orig=model.W1[idx]
        model.W1[idx]=orig+h; lp=model.loss(y,model.forward(X))
        model.W1[idx]=orig-h; lm=model.loss(y,model.forward(X))
        model.W1[idx]=orig
        num[idx]=(lp-lm)/(2*h); it.iternext()

    err = np.linalg.norm(analytic-num)/(np.linalg.norm(analytic)+np.linalg.norm(num)+1e-15)
    print(f"Relatif hata: {err:.2e}  →  {'✓ BAŞARILI' if err<1e-4 else '✗ HATA'}")
    return err

np.random.seed(0)
test_model = MLP(2,3,1,.1)
gradient_check(test_model, X_xor, y_xor)


## 6. Mini Autograd Motoru <a id='6'></a>

PyTorch'un temelindeki mantık: her işlem bir backward fonksiyonu kaydeder.

In [ ]:
class Value:
    def __init__(self,data,_ch=(),_op='',label=''):
        self.data=float(data); self.grad=0.; self._backward=lambda:None
        self._prev=set(_ch); self._op=_op; self.label=label
    def __repr__(self): return f"Value({self.data:.3f}, grad={self.grad:.3f})"
    def __add__(self,o):
        o=o if isinstance(o,Value) else Value(o)
        out=Value(self.data+o.data,(self,o),'+')
        def _bw(): self.grad+=out.grad; o.grad+=out.grad
        out._backward=_bw; return out
    def __mul__(self,o):
        o=o if isinstance(o,Value) else Value(o)
        out=Value(self.data*o.data,(self,o),'*')
        def _bw(): self.grad+=out.grad*o.data; o.grad+=out.grad*self.data
        out._backward=_bw; return out
    def __pow__(self,e):
        out=Value(self.data**e,(self,),f'**{e}')
        def _bw(): self.grad+=out.grad*e*(self.data**(e-1))
        out._backward=_bw; return out
    def __sub__(self,o): return self+(o*-1)
    def __neg__(self): return self*-1
    def __radd__(self,o): return self+o
    def __rmul__(self,o): return self*o
    def sigmoid(self):
        s=1/(1+np.exp(-np.clip(self.data,-500,500)))
        out=Value(s,(self,),'sigmoid')
        def _bw(): self.grad+=out.grad*s*(1-s)
        out._backward=_bw; return out
    def backward(self):
        topo=[]; visited=set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self); self.grad=1.
        for node in reversed(topo): node._backward()

# Demo
w=Value(2.,label='w'); x=Value(3.,label='x'); y=Value(5.,label='y')
L = (w*x-y)**2
L.backward()
print(f"L = (w·x − y)²  →  L={L.data}")
print(f"dL/dw={w.grad} (beklenen: {2*(2.*3.-5.)*3.})")
print(f"dL/dx={x.grad} (beklenen: {2*(2.*3.-5.)*2.})")


## 7. Vanishing & Exploding Gradient <a id='7'></a>

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))

# Vanishing
ax1=axes[0]; n=np.arange(1,21)
ax1.semilogy(n,0.25**n,'b-o',ms=5,lw=2,label='Sigmoid (maks 0.25)')
ax1.semilogy(n,np.ones(20),'r-^',ms=5,lw=2,label='ReLU (z>0: 1.0)')
ax1.fill_between(n,0,1e-7,alpha=.1,color='red')
ax1.set_title('Vanishing Gradient',fontweight='bold')
ax1.set_xlabel('Katman Sayısı'); ax1.set_ylabel('Gradyan (log)')
ax1.legend(); ax1.grid(True,alpha=.3)

# Exploding + Clipping
ax2=axes[1]; steps=np.arange(20)
raw=[1.8**i for i in steps]
clipped=[min(g,5.) for g in raw]
ax2.plot(raw,'r-o',lw=2,ms=5,label='Ham gradyan (w=1.8)')
ax2.plot(clipped,'b-s',lw=2,ms=5,label='Clipped (max=5)')
ax2.axhline(5,color='blue',ls='--',alpha=.6,label='Clip sınırı')
ax2.set_title('Exploding Gradient + Clipping',fontweight='bold')
ax2.set_xlabel('Adım'); ax2.set_ylabel('Gradyan'); ax2.legend(); ax2.grid(True,alpha=.3)
ax2.set_ylim(0,30)

plt.suptitle('Backpropagation Gradient Sorunları',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()
print(f"20 katman Sigmoid gradyanı: {0.25**20:.2e}")
print(f"20 adım Exploding gradyanı: {1.8**20:.2e}")


## 8. Alıştırmalar <a id='10'></a>

**1.** `Value` sınıfına `relu()` ve `log()` metodunu ekle ve XOR problemini çöz.

**2.** 3 katmanlı MLP için tüm gradyanları kağıt üzerinde türet ve sayısal gradyanla karşılaştır.

**3.** Gradient clipping'i norm bazlı uygula: `total_norm = sqrt(Σ||g||²)`, `scale = max_norm/total_norm`.

**4.** `MLP` sınıfına eğitim boyunca W1 gradyan normunu kaydet ve grafik çiz — vanishing görebiliyor musun?

**5.** PyTorch'ta `requires_grad=True` kullanarak aynı 2 katmanlı ağı yaz ve gradyanları karşılaştır.

---
**Sonraki Modül:** Modül 05 — Regularization (Dropout, Batch Norm, L1/L2, Early Stopping)
